# CNNs for chest X-ray and retinal images: worked solution

Worked solution for the exercise in the lesson **[CNNs for chest X-ray and retinal images](https://ramkumarmd.com/medical-ai/deep-learning-imaging/cnns-medical-imaging)** (*Deep learning & imaging* track, Medical AI course by Dr. Ram Kumar G). Try the [exercise notebook](02-cnns-medical-imaging.ipynb) first. Other answers can be just as good: the written parts show one reasonable answer, not the only one.

**In short:** Convolutional neural networks (CNNs) scan an image with small learned filters, so a feature such as a lesion's edge is recognised wherever it appears. This makes them far better than pixel-by-pixel models for medical images, and CNNs underpin many published systems for diabetic retinopathy, skin lesions and chest X-rays. They also learn whatever separates the training labels, including scanner markers or hospital labels, so a model can look excellent at its own hospital and fail at another.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Setup

In [ ]:
%pip install -q medmnist

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### Training a small CNN

Synthetic scans, a two-layer CNN and its test accuracy. Takes a few seconds on a CPU.

In [ ]:
import numpy as np
import torch
from torch import nn

def make_images(n, seed, marker_with_lesion=0.0):
    """Synthetic 28x28 'scans': noisy background; half contain a faint round lesion at a random place.
    marker_with_lesion: chance that lesion images also carry a bright corner marker (a shortcut)."""
    rng = np.random.default_rng(seed)
    yy, xx = np.mgrid[:28, :28]
    images = rng.normal(0, 1, (n, 28, 28))
    labels = rng.integers(0, 2, n)
    for i in np.where(labels == 1)[0]:
        cy, cx, r = rng.integers(6, 22), rng.integers(6, 22), rng.uniform(2.5, 4.0)
        images[i] += 1.0 * ((yy - cy) ** 2 + (xx - cx) ** 2 < r ** 2)
        if rng.random() < marker_with_lesion:
            images[i, :3, :3] += 4.0                     # e.g. a portable-scanner label
    return torch.tensor(images, dtype=torch.float32).unsqueeze(1), torch.tensor(labels)

class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),      # 28 -> 14
            nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),     # 14 -> 7
        )
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(16 * 7 * 7, 2))
    def forward(self, x):
        return self.head(self.features(x))

def train(model, X, y, epochs=8, seed=0):
    torch.manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    for _ in range(epochs):
        for idx in torch.randperm(len(X)).split(64):
            opt.zero_grad()
            nn.functional.cross_entropy(model(X[idx]), y[idx]).backward()
            opt.step()
    return model

def accuracy(model, X, y):
    with torch.no_grad():
        return (model(X).argmax(1) == y).float().mean().item()

X_train, y_train = make_images(4000, seed=1)
X_test, y_test = make_images(1000, seed=2)
torch.manual_seed(0)
cnn = train(SmallCNN(), X_train, y_train)
print(f"CNN test accuracy: {accuracy(cnn, X_test, y_test):.3f}")
print(f"Parameters: {sum(p.numel() for p in cnn.parameters()):,}")

# CNN test accuracy: 0.946
# Parameters: 2,818

### Why convolution matters

Continuing from the previous block: logistic regression on the raw pixels, for comparison.

In [ ]:
from sklearn.linear_model import LogisticRegression

flat = LogisticRegression(max_iter=2000).fit(X_train.reshape(len(X_train), -1).numpy(), y_train.numpy())
print(f"Logistic regression on raw pixels: "
      f"{flat.score(X_test.reshape(len(X_test), -1).numpy(), y_test.numpy()):.3f}")

# Logistic regression on raw pixels: 0.738

### Shortcut learning

Continuing from the first block: train where the marker goes with lesions, test with and without it.

In [ ]:
X_dev, y_dev = make_images(4000, seed=3, marker_with_lesion=0.9)
torch.manual_seed(0)
shortcut_cnn = train(SmallCNN(), X_dev, y_dev)
X_same, y_same = make_images(1000, seed=4, marker_with_lesion=0.9)
X_new, y_new = make_images(1000, seed=5, marker_with_lesion=0.0)
print(f"Same hospital (marker present): {accuracy(shortcut_cnn, X_same, y_same):.3f}")
print(f"New hospital (no marker):       {accuracy(shortcut_cnn, X_new, y_new):.3f}")

# Same hospital (marker present): 0.984
# New hospital (no marker):       0.813

## Exercise

**Goal:** Train CNNs on synthetic and real medical images, and test them for shortcut learning.

**Data:** Synthetic scans in the notebook; MedMNIST PneumoniaMNIST and RetinaMNIST (optional download) (Generated in the notebook; MedMNIST under CC BY 4.0)

### Step 1

Reproduce the CNN, pixel baseline and shortcut experiments, and vary the marker rate from 0.5 to 1.0.

In [ ]:
print(f"CNN {accuracy(cnn, X_test, y_test):.3f}; pixel logistic regression "
      f"{flat.score(X_test.reshape(len(X_test), -1).numpy(), y_test.numpy()):.3f}")
print("marker rate   same hospital   new hospital")
for rate in [0.5, 0.7, 0.9, 1.0]:
    Xd, yd = make_images(4000, seed=3, marker_with_lesion=rate)
    torch.manual_seed(0)
    m = train(SmallCNN(), Xd, yd)
    Xs, ys = make_images(1000, seed=4, marker_with_lesion=rate)
    print(f"{rate:>11.1f}   {accuracy(m, Xs, ys):>13.3f}   {accuracy(m, X_new, y_new):>12.3f}")

The more reliably the marker accompanies lesions, the better the model looks at its own hospital and the worse it
does where the marker never appears: at a rate of 1.0, the marker alone predicts the label perfectly.

### Step 2

Mask the top-left corner of every image before training the shortcut model; how much does new-hospital accuracy recover?

In [ ]:
def mask_corner(X):
    X = X.clone()
    X[:, :, :3, :3] = 0.0                     # remove the region where the marker appears
    return X

torch.manual_seed(0)
masked_cnn = train(SmallCNN(), mask_corner(X_dev), y_dev)
print(f"Shortcut model, new hospital:        {accuracy(shortcut_cnn, X_new, y_new):.3f}")
print(f"Corner-masked model, new hospital:   {accuracy(masked_cnn, mask_corner(X_new), y_new):.3f}")
print(f"Model trained without markers:       {accuracy(cnn, X_new, y_new):.3f}")

Masking the corner removes the shortcut, so the network has to learn the lesion itself, and new-hospital accuracy
recovers to the level of a model trained on clean images. In real images the shortcut is rarely so easy to locate
(text labels, positioning, image processing), which is why external testing matters more than any single fix.

### Step 3

Train the same CNN on PneumoniaMNIST and report AUROC, sensitivity and specificity on its test set.

In [ ]:
from medmnist import PneumoniaMNIST
from sklearn.metrics import confusion_matrix, roc_auc_score

def load_medmnist(cls, split):
    d = cls(split=split, download=True, root="data", size=28)
    X = torch.tensor(d.imgs, dtype=torch.float32).unsqueeze(1) / 255.0
    X = (X - 0.5) / 0.25                                   # roughly standardise intensities
    return X, torch.tensor(d.labels.ravel(), dtype=torch.long)

Xp_tr, yp_tr = load_medmnist(PneumoniaMNIST, "train")
Xp_te, yp_te = load_medmnist(PneumoniaMNIST, "test")
print(f"PneumoniaMNIST: {len(Xp_tr)} training and {len(Xp_te)} test images; pneumonia in "
      f"{yp_tr.float().mean():.0%} of training and {yp_te.float().mean():.0%} of test images")

torch.manual_seed(0)
pneu = train(SmallCNN(), Xp_tr, yp_tr, epochs=8)

def report(model, X, y, label):
    with torch.no_grad():
        p = model(X).softmax(1)[:, 1].numpy()
    tn, fp, fn, tp = confusion_matrix(y.numpy(), p >= 0.5).ravel()
    print(f"{label:<28} AUROC {roc_auc_score(y.numpy(), p):.3f}  sensitivity {tp / (tp + fn):.3f}  "
          f"specificity {tn / (tn + fp):.3f}")

report(pneu, Xp_te, yp_te, "SmallCNN")

PneumoniaMNIST is a 28x28 version of a public paediatric chest X-ray dataset (MedMNIST, CC BY 4.0). Typical of
this dataset, sensitivity is very high and specificity much lower at the default threshold (0.995 and 0.538 here),
and the test set has a different class balance from the training set (printed above). These are tiny, downsampled images: the numbers say nothing about diagnostic
performance on real radiographs.

### Step 4

Add a third convolutional layer and data augmentation (random flips and shifts) and compare.

In [ ]:
class DeeperCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),        # 28 -> 14
            nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),       # 14 -> 7
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1))
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(32, 2))
    def forward(self, x):
        return self.head(self.features(x))

def augment(X, rng):
    X = X.clone()
    flip = torch.tensor(rng.random(len(X)) < 0.5)
    X[flip] = X[flip].flip(-1)                                             # random horizontal flip
    shifts = rng.integers(-2, 3, size=(len(X), 2))
    return torch.stack([torch.roll(x, shifts=(int(dy), int(dx)), dims=(1, 2)) for x, (dy, dx) in zip(X, shifts)])

def train_augmented(model, X, y, epochs=8, seed=0):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    for _ in range(epochs):
        Xa = augment(X, rng)                                               # a new random version each epoch
        for idx in torch.randperm(len(X)).split(64):
            opt.zero_grad()
            nn.functional.cross_entropy(model(Xa[idx]), y[idx]).backward()
            opt.step()
    return model

torch.manual_seed(0)
deeper = train(DeeperCNN(), Xp_tr, yp_tr, epochs=8)
report(deeper, Xp_te, yp_te, "Three conv layers")
torch.manual_seed(0)
deeper_aug = train_augmented(DeeperCNN(), Xp_tr, yp_tr, epochs=8)
report(deeper_aug, Xp_te, yp_te, "Three conv layers + augment")

The deeper network improves AUROC (0.928 to 0.947) and, more usefully, specificity (0.538 to 0.756), at a small cost
in sensitivity; augmentation adds a little more specificity. Compare on AUROC and on the sensitivity/specificity
balance, not on one number. Whether flips are a
sensible augmentation is itself a clinical question: a horizontally flipped chest X-ray puts the heart on the wrong
side, which never happens in real images (except in dextrocardia), so small shifts are the safer augmentation here.

---

Lesson: [https://ramkumarmd.com/medical-ai/deep-learning-imaging/cnns-medical-imaging](https://ramkumarmd.com/medical-ai/deep-learning-imaging/cnns-medical-imaging). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.